# 🔍 Scout Agent Test - Job Discovery Engine

**Market Mapper: Real-time Job Discovery using Mistral Agents**

This notebook tests the Scout agent that:
- Uses **web_search_preview** to find current job listings
- Matches opportunities to candidate's "Strong Matches" from resume analysis
- Performs multi-source searches (internal, competitors, skill-based)
- Outputs structured discovery results with direct links

## 📦 Step 0: Install Dependencies

**Important**: After running the installation cell below, if you get a `ModuleNotFoundError`, restart the Jupyter kernel (Kernel → Restart) and run the cells again.

In [1]:
# Install mistralai package
%pip install mistralai --upgrade -q

# Verify installation
try:
    import mistralai
    print('✅ mistralai package installed successfully')
    print(f'   Version: {mistralai.__version__ if hasattr(mistralai, "__version__") else "unknown"}')
except ImportError:
    print('❌ Installation failed. Please restart kernel and try again.')
    raise


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ mistralai package installed successfully
   Version: 1.12.4


## 🔑 Step 1: Configure API Key

In [2]:
import os
from dotenv import load_dotenv
from mistralai import Mistral

load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
client = Mistral(api_key=api_key)
print("Mistral client initialised.")

Mistral client initialised.


## 🤖 Step 2: Create the Scout Agent

The Scout agent is configured with `web_search_preview` tool for real-time job discovery.

In [3]:
import os
from mistralai import Mistral

SCOUT_MODEL = "mistral-large-latest" 

# 2. Refined System Prompt (Optimized for Agentic Reasoning)
SCOUT_SYSTEM_PROMPT = """You are the "Market Mapper" Scout, a high-speed recruitment intelligence agent. 
Your purpose is to bridge the gap between a candidate's current profile and the real-time 2026 job market.

**IMPORTANT: You have access to the 'web_search' tool. You MUST use it to find current job listings.**

Operational Protocol:
1. **JSON Data**: The candidate's resume JSON will be provided in the user message. Analyze it directly from the text - it's not a file, it's embedded in the prompt.

2. **Web Search Tool**: You HAVE the 'web_search' tool available. You MUST use it to:
   - Search for jobs at the target company (e.g., "Google careers data scientist")
   - Find similar roles at competitors (e.g., "data scientist jobs at Microsoft Amazon")
   - Search for roles matching candidate skills (e.g., "Python machine learning jobs 2026")
   **Use the web_search tool for ALL searches - do not say you cannot search the web.**

3. **Match Logic**: 
   - High: >80% overlap with candidate's Strong Matches.
   - Medium: 50-80% overlap.
   - Strategic Pivot: Uses candidate's unique niche skills despite low overall match.

4. **Output**: Provide results in a Markdown table with columns:
   - Job Title
   - Company
   - Match Level (High/Medium/Strategic Pivot)
   - Why it Matches (reference specific skills from the JSON)
   - Source Link (actual URL from web search)

5. **Constraints**: Stay objective. Focus on discovery. Provide actual, verified URLs from your web searches."""

# 3. Create the Agent using the Agents API
# This 'attaches' the web search tool to the model automatically.
scout_agent = client.beta.agents.create(
    name="Market Mapper Scout",
    description="Finds similar roles and competitor opportunities based on resume data.",
    model=SCOUT_MODEL,
    instructions=SCOUT_SYSTEM_PROMPT,
    tools=[{
        "type": "web_search" # This activates the 'web_search_preview' capability
    }]
)

print(f'✅ Scout Agent configured')
print(f'   Model: {SCOUT_MODEL}')
print(f'   Agent ID: {scout_agent.id}') # Use this ID in your chat.complete calls
print(f'   Tool: web_search enabled')

✅ Scout Agent configured
   Model: mistral-large-latest
   Agent ID: ag_019ca63ed51c704c9f47ad9bf62bf934
   Tool: web_search enabled


## 📋 Step 3: Prepare Input Data

Provide the Auditor's analysis JSON (resume and JD analysis) and target company.

In [4]:
import json
from pathlib import Path

# Example Job Description (fallback if no JD is loaded from file)
auditor_jd_json = {
    "job_title": "Senior Data Scientist",
    "company": "Google",
    "required_skills": ["Python", "Machine Learning", "Data Analysis", "SQL"],
    "preferred_skills": ["Apache Spark", "AWS", "Team Management"],
    "description": "Looking for senior data scientist with experience in machine learning and data pipelines"
}

# Initialize resume variable
auditor_resume_json = None

# File containing the structured resume
auditor_result_file = "resume_llm_structured.json"

# Try loading the resume file
if Path(auditor_result_file).exists():
    try:
        with open(auditor_result_file, "r", encoding="utf-8") as f:
            auditor_resume_json = json.load(f)

        print("✅ Resume JSON loaded successfully")

    except Exception as e:
        print(f"⚠️ Error loading resume file: {e}")
        print("Using example JD only.")

else:
    print(f"ℹ️ File '{auditor_result_file}' not found.")
    print("Place your structured resume JSON in the project folder.")

# Safety check
if auditor_resume_json is None:
    print("⚠️ WARNING: Resume JSON not loaded.")

# Target company for job search
target_company = "Google"

# Debug info
print("\nDebug Info:")
print("Resume loaded:", auditor_resume_json is not None)
print("JD loaded:", auditor_jd_json is not None)

# Example of accessing resume data
if auditor_resume_json:
    skills = auditor_resume_json["sections"]["skills"]
    experience = auditor_resume_json["sections"]["experience"]

    print("\nTop Skills:", skills[:5])
    print("Number of jobs:", len(experience))

✅ Resume JSON loaded successfully

Debug Info:
Resume loaded: True
JD loaded: True

Top Skills: ['Machine Learning', 'Data Analysis', 'Database Administration', 'Python', 'R']
Number of jobs: 2


## 🔍 Step 4: Execute Opportunity Scan

The Scout agent will perform multi-source searches and match opportunities.

In [6]:
# Get variables from the appropriate scope
auditor_resume_json = locals().get('auditor_resume_json') or globals().get('auditor_resume_json')
auditor_jd_json = locals().get('auditor_jd_json') or globals().get('auditor_jd_json')

# Construct the discovery prompt
# The LLM will analyze the JSON and use web_search_preview to find matching opportunities
resume_json_str = json.dumps(auditor_resume_json, ensure_ascii=False)
jd_json_str = json.dumps(auditor_jd_json, ensure_ascii=False)

discovery_prompt = f"""I am initiating an "Opportunity Scan" for the following candidate.

**Target Company:** {target_company}

**Candidate Resume JSON (START):**
{resume_json_str}
**Candidate Resume JSON (END)**

**Original Job Description JSON (START):**
{jd_json_str}
**Original Job Description JSON (END)**

**Your Mission:**


1. **Analyze the Resume JSON** (provided below): Review the candidate's profile including:
   - Their skills (from the skills array and project technologies)
   - Their experience and achievements
   - Their projects and technologies used
   - Their education and certifications
   - Their strengths and expertise areas

2. **Identify Strong Matches**: Based on your analysis, identify the candidate's core competencies and strongest skills that align with the target role.

3. **Perform Web Searches** using your 'web_search' tool (you have this tool - use it now):
   - **Search A (Internal Depth)**: Search for 1 open role at {target_company} that matches the candidate's profile. Use search terms like: "{target_company} careers [role title]" or "{target_company} jobs [key skills]"
   
   - **Search B (Market Breadth)**: Search for 2 similar roles at competitors or high-growth startups. Use search terms like: "jobs like [role title] at [competitor]" or "[key skills] jobs 2026"
   
   - **Search C (Skill-Based Discovery)**: Search for roles prioritizing the candidate's top skills. Extract the top 3-5 most relevant skills from the resume JSON and search for: "companies hiring [skill1] [skill2] [skill3] 2026"

4. **Compare & Map**: For each job found, analyze how it matches the candidate's profile from the Resume JSON, referencing specific skills, experience, or projects.

**Deliverable:**
Provide a Markdown table with the following columns:
- **Job Title**
- **Company** (if available)
- **Match Level** (High: >80% match, Medium: 50-80%, Strategic Pivot: specialized niche)
- **Why it Matches** (1-2 sentences referencing specific skills/experience from the Resume JSON)
- **Source Link** (direct URL to the job posting)

**CRITICAL INSTRUCTIONS:** 
- **You HAVE the 'web_search' tool - USE IT NOW to find job listings. Do not say you cannot search the web.**
- **The Resume JSON is located ABOVE in this message between "Candidate Resume JSON (START)" and "Candidate Resume JSON (END)" - it is embedded in this message, NOT a separate file.**
- **The Job Description JSON is located ABOVE between "Original Job Description JSON (START)" and "Original Job Description JSON (END)" - read it from this message.**
- **Both JSON objects are provided in this message - scroll up and read them.**
- Use your web_search tool to find actual current job postings.
- Reference specific skills, technologies, or achievements from the JSON in your matching analysis.
- Stay objective. Focus on job discovery only. Do not discuss rejection reasons.
- Ensure all job links are current and verified URLs from your web searches.
"""

print("🚀 Executing Opportunity Scan...")
print("=" * 60)

# Debug: Show that JSON is included
print(f"📋 Prompt includes:")
print(f"   - Resume JSON length: {len(resume_json_str)} characters")
print(f"   - JD JSON length: {len(jd_json_str)} characters")
print(f"   - Target Company: {target_company}")
print(f"   - Total prompt length: {len(discovery_prompt)} characters")
print()
print("📄 Preview of JSON in prompt (first 200 chars):")
print(f"   Resume JSON: {resume_json_str[:200]}...")
print(f"   JD JSON: {jd_json_str[:200]}...")
print()

print("📡 Sending request to Mistral AI with native web search...")
print("⏳ This may take 30-60 seconds due to live web browsing...")

# The FIX: Use the 'web_search' tool type specifically
try:
   response = client.beta.conversations.start(
        agent_id=scout_agent.id,
        inputs=discovery_prompt
   )

   print(f"✅ Response received!")

except Exception as e:
    print(f"❌ API Error: {e}")

🚀 Executing Opportunity Scan...
📋 Prompt includes:
   - Resume JSON length: 3362 characters
   - JD JSON length: 305 characters
   - Target Company: Google
   - Total prompt length: 6531 characters

📄 Preview of JSON in prompt (first 200 chars):
   Resume JSON: {"name": "John Doe", "contact": {"email": "John.doe@example.com", "location": "New York, USA", "linkedin": "linkedin.com/in/johndoe", "phone": "123-456-7890", "website": "https://www.johndoe.com", "po...
   JD JSON: {"job_title": "Senior Data Scientist", "company": "Google", "required_skills": ["Python", "Machine Learning", "Data Analysis", "SQL"], "preferred_skills": ["Apache Spark", "AWS", "Team Management"], "...

📡 Sending request to Mistral AI with native web search...
⏳ This may take 30-60 seconds due to live web browsing...
❌ API Error: API error occurred: Status 429. Body: {"detail":"web_search rate limit reached."}


## 📊 Step 5: Retrieve and Display Results

In [7]:
import time
from IPython.display import Markdown, display

# Display the response from Step 4
try:
    if 'response' in locals():
        print("📊 Scout Agent Response:")
        print("=" * 60)
        
        # Handle the beta.conversations.start() response format
        if hasattr(response, 'messages'):
            # If response has messages, get the last assistant message
            assistant_messages = [msg for msg in response.messages if hasattr(msg, 'role') and msg.role == "assistant"]
            if assistant_messages:
                message = assistant_messages[-1]
            else:
                message = response.messages[-1] if response.messages else None
        elif hasattr(response, 'content'):
            message = response
        elif hasattr(response, 'choices'):
            # Chat completion format
            message = response.choices[0].message
        else:
            message = response
        
        # Display the response content
        if message:
            if hasattr(message, 'content'):
                content = message.content
                if isinstance(content, list):
                    # Handle structured content
                    for item in content:
                        if hasattr(item, 'type') and item.type == "text":
                            display(Markdown(item.text if hasattr(item, 'text') else str(item)))
                        elif hasattr(item, 'text'):
                            display(Markdown(item.text))
                        else:
                            print(f"Content type: {type(item)}")
                            print(str(item))
                elif content:
                    display(Markdown(str(content)))
                else:
                    print("Response content:")
                    print(json.dumps(message.model_dump() if hasattr(message, 'model_dump') else str(message), indent=2))
            elif hasattr(message, 'text'):
                display(Markdown(message.text))
            else:
                # Try to extract text from response
                response_str = str(message)
                if response_str:
                    display(Markdown(response_str))
                else:
                    print(json.dumps(message.model_dump() if hasattr(message, 'model_dump') else str(message), indent=2))
        
        # Check for tool calls if available
        if hasattr(message, 'tool_calls') and message.tool_calls:
            print("\n🔧 Tool Calls Used:")
            for tool_call in message.tool_calls:
                print(f"   - {tool_call.type if hasattr(tool_call, 'type') else 'N/A'}")
        
        print("=" * 60)
        
        # Save results to file
        output_file = f"scout_results_{int(time.time())}.json"
        result_data = {
            "agent_id": scout_agent.id if 'scout_agent' in locals() else None,
            "target_company": target_company,
            "response_content": message.content if hasattr(message, 'content') else (message.text if hasattr(message, 'text') else str(message)),
            "response_full": message.model_dump() if hasattr(message, 'model_dump') else str(message),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result_data, f, indent=2, default=str)
        
        print(f"\n✅ Results saved to: {output_file}")
    else:
        print("⚠️  No response available. Run Step 4 first to execute the opportunity scan.")
        
except Exception as e:
    print(f"❌ Error displaying response: {e}")
    import traceback
    traceback.print_exc()
    print("\n💡 Tip: Make sure you've run Step 4 first to get the response.")

⚠️  No response available. Run Step 4 first to execute the opportunity scan.
